# NLPMini_EX — Hindi Multi-Label Emotion Detection

Train a model on **EmoInHindi**, evaluate it, and export a ZIP containing the trained model for later predictions.

**Colab setup:** Upload `EmoInHindi.zip` to the Files panel so it appears at `/content/EmoInHindi.zip`. The notebook also works if you have already extracted `LREC_EmoInHindi.csv` under `/content/EmoInHindi/`.

This concise baseline predicts emotions from each utterance alone. Splits are grouped by `dialogueId` to reduce dialogue leakage. The dataset README specifies research use only.

## 1. Install and import libraries

In [ ]:
!pip -q install scikit-learn joblib pandas matplotlib

import os, glob, zipfile, json, joblib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    hamming_loss, jaccard_score, classification_report, precision_recall_curve)
from IPython.display import display
warnings.filterwarnings('ignore')
RANDOM_STATE = 42

## 2. Load and prepare EmoInHindi

The CSV contains `utterance`, `emotions`, and `dialogueId`. Emotion labels are comma-separated. The provided CSV contains **17 distinct labels** including `neutral`, so labels are detected automatically.

In [ ]:
DATA_DIR = '/content/EmoInHindi'
CSV_PATH = os.path.join(DATA_DIR, 'LREC_EmoInHindi.csv')
ZIP_PATH = '/content/EmoInHindi.zip'

if not os.path.exists(CSV_PATH):
    if os.path.exists(ZIP_PATH):
        with zipfile.ZipFile(ZIP_PATH) as z:
            z.extractall('/content')
    else:
        matches = glob.glob('/content/**/LREC_EmoInHindi.csv', recursive=True)
        if matches:
            CSV_PATH = matches[0]
        else:
            raise FileNotFoundError('Upload EmoInHindi.zip to /content or extract LREC_EmoInHindi.csv under /content/EmoInHindi/')
if not os.path.exists(CSV_PATH):
    matches = glob.glob('/content/**/LREC_EmoInHindi.csv', recursive=True)
    if not matches:
        raise FileNotFoundError('Could not find LREC_EmoInHindi.csv after extraction.')
    CSV_PATH = matches[0]

df = pd.read_csv(CSV_PATH).dropna(subset=['utterance', 'emotions', 'dialogueId']).copy()
df['utterance'] = df['utterance'].astype(str).str.strip()
df = df[df['utterance'].ne('')].reset_index(drop=True)
df['emotion_list'] = df['emotions'].astype(str).apply(lambda s: [x.strip() for x in s.split(',') if x.strip()])
labels = sorted({lab for row in df['emotion_list'] for lab in row})
label_to_id = {lab: i for i, lab in enumerate(labels)}
Y = np.zeros((len(df), len(labels)), dtype=np.uint8)
for i, labs in enumerate(df['emotion_list']):
    for lab in labs:
        Y[i, label_to_id[lab]] = 1

print(f'Utterances: {len(df):,} | Dialogues: {df["dialogueId"].nunique():,} | Labels: {len(labels)}')
print('Labels:', ', '.join(labels))
display(pd.DataFrame({'emotion': labels, 'positive_examples': Y.sum(axis=0)}).sort_values('positive_examples', ascending=False))

## 3. Split by dialogue

Utterances from the same dialogue stay in one split, reducing train/test leakage.

In [ ]:
groups = df['dialogueId'].to_numpy()
indices = np.arange(len(df))
split1 = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=RANDOM_STATE)
trainval_idx, test_idx = next(split1.split(indices, groups=groups))
split2 = GroupShuffleSplit(n_splits=1, test_size=0.117647, random_state=RANDOM_STATE)
train_rel, val_rel = next(split2.split(trainval_idx, groups=groups[trainval_idx]))
train_idx, val_idx = trainval_idx[train_rel], trainval_idx[val_rel]
X_text = df['utterance'].to_numpy()
X_train, X_val, X_test = X_text[train_idx], X_text[val_idx], X_text[test_idx]
Y_train, Y_val, Y_test = Y[train_idx], Y[val_idx], Y[test_idx]
print(f'Train: {len(train_idx):,} ({len(train_idx)/len(df):.1%})')
print(f'Validation: {len(val_idx):,} ({len(val_idx)/len(df):.1%})')
print(f'Test: {len(test_idx):,} ({len(test_idx)/len(df):.1%})')

## 4. Train the model

A fast non-transformer baseline: word + character TF-IDF with one-vs-rest Linear SVM. It predicts from the current utterance only; it does not use dialogue context or predict intensity.

In [ ]:
vectorizer = FeatureUnion([
    ('word', TfidfVectorizer(analyzer='word', ngram_range=(1, 2), max_features=40000, min_df=2, sublinear_tf=True)),
    ('char', TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), max_features=30000, min_df=2, sublinear_tf=True))
])
print('Building TF-IDF features...')
X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)
X_test_vec = vectorizer.transform(X_test)
print('Feature matrix:', X_train_vec.shape)
model = OneVsRestClassifier(LinearSVC(C=1.0, class_weight='balanced', max_iter=3000, random_state=RANDOM_STATE))
print('Training one classifier per emotion...')
model.fit(X_train_vec, Y_train)
val_scores = model.decision_function(X_val_vec)
test_scores = model.decision_function(X_test_vec)
print('Training complete.')

## 5. Tune thresholds on validation data

Thresholds are selected using validation data only to optimize F1 for each emotion. The test set is not used for tuning.

In [ ]:
thresholds = np.zeros(len(labels), dtype=float)
for j in range(len(labels)):
    y_true, scores = Y_val[:, j], val_scores[:, j]
    if y_true.sum() < 3 or y_true.sum() == len(y_true):
        thresholds[j] = 0.0
        continue
    precision, recall, candidates = precision_recall_curve(y_true, scores)
    if len(candidates) == 0:
        thresholds[j] = 0.0
        continue
    denom = precision[:-1] + recall[:-1]
    f1_values = np.divide(2 * precision[:-1] * recall[:-1], denom, out=np.zeros_like(denom), where=denom > 0)
    thresholds[j] = candidates[int(np.argmax(f1_values))]
Y_pred = (test_scores >= thresholds).astype(np.uint8)
print('Validation threshold tuning complete.')

## 6. Test metrics

Label-wise accuracy counts every present/absent emotion decision. Subset accuracy requires all labels for a sentence to match exactly. Read F1 and recall alongside accuracy.

In [ ]:
def calculate_metrics(y_true, y_pred):
    return {
        'Label-wise Accuracy': accuracy_score(y_true.ravel(), y_pred.ravel()),
        'Subset Accuracy / Exact Match': accuracy_score(y_true, y_pred),
        'Micro Precision': precision_score(y_true, y_pred, average='micro', zero_division=0),
        'Macro Precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'Weighted Precision': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'Micro Recall / Sensitivity': recall_score(y_true, y_pred, average='micro', zero_division=0),
        'Macro Recall / Sensitivity': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'Weighted Recall': recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'Micro F1': f1_score(y_true, y_pred, average='micro', zero_division=0),
        'Macro F1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'Weighted F1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'Hamming Loss (lower is better)': hamming_loss(y_true, y_pred),
        'Sample Jaccard': jaccard_score(y_true, y_pred, average='samples', zero_division=0),
    }
metrics = calculate_metrics(Y_test, Y_pred)
display(pd.DataFrame({'Metric': list(metrics.keys()), 'Score': [round(v, 4) for v in metrics.values()]}))
print('\nPer-emotion classification report:')
print(classification_report(Y_test, Y_pred, target_names=labels, zero_division=0, digits=4))

## 7. Per-emotion F1

In [ ]:
per_label_f1 = f1_score(Y_test, Y_pred, average=None, zero_division=0)
plot_df = pd.DataFrame({'Emotion': labels, 'F1': per_label_f1}).sort_values('F1')
plt.figure(figsize=(9, 6))
plt.barh(plot_df['Emotion'], plot_df['F1'])
plt.xlabel('F1 score')
plt.title('Test F1 by emotion')
plt.xlim(0, 1)
plt.tight_layout()
plt.show()

## 8. Export the trained model as a ZIP

The ZIP contains the fitted vectorizer, classifiers, thresholds, label list, prediction helper and README. Reuse it without retraining.

In [ ]:
OUTPUT_DIR = '/content/EmoInHindi_model'
os.makedirs(OUTPUT_DIR, exist_ok=True)
bundle = {'vectorizer': vectorizer, 'model': model, 'labels': labels,
          'thresholds': thresholds.tolist(),
          'model_type': 'TF-IDF word+character + OneVsRest LinearSVC'}
joblib.dump(bundle, os.path.join(OUTPUT_DIR, 'model_bundle.joblib'), compress=3)
with open(os.path.join(OUTPUT_DIR, 'labels.json'), 'w', encoding='utf-8') as f:
    json.dump(labels, f, ensure_ascii=False, indent=2)
with open(os.path.join(OUTPUT_DIR, 'thresholds.json'), 'w', encoding='utf-8') as f:
    json.dump(dict(zip(labels, thresholds.tolist())), f, ensure_ascii=False, indent=2)

predict_lines = [
'import joblib', 'import numpy as np', '',
'def load_emotion_model(path="model_bundle.joblib"):', '    return joblib.load(path)', '',
'def predict_emotions(text, bundle):', '    X = bundle["vectorizer"].transform([text])',
'    scores = bundle["model"].decision_function(X)[0]',
'    thresholds = np.asarray(bundle["thresholds"])',
'    selected = [label for label, score, threshold in zip(bundle["labels"], scores, thresholds) if score >= threshold]',
'    return {"emotions": selected, "scores": dict(zip(bundle["labels"], scores.tolist()))}', '',
'if __name__ == "__main__":', '    bundle = load_emotion_model()',
'    print(predict_emotions("मैं बहुत दुखी और परेशान हूँ।", bundle))'
]
with open(os.path.join(OUTPUT_DIR, 'predict.py'), 'w', encoding='utf-8') as f:
    f.write('\n'.join(predict_lines))
readme = ('EmoInHindi multi-label emotion model\n\nFiles: model_bundle.joblib, labels.json, thresholds.json, predict.py\n\n'
          'Usage: pip install scikit-learn joblib numpy\n'
          'from predict import load_emotion_model, predict_emotions\n'
          'bundle = load_emotion_model("model_bundle.joblib")\n'
          'print(predict_emotions("मैं बहुत दुखी हूँ।", bundle)["emotions"])\n\n'
          'Research use only. Predicts from a single utterance; no dialogue context or intensity prediction.')
with open(os.path.join(OUTPUT_DIR, 'README.txt'), 'w', encoding='utf-8') as f:
    f.write(readme)
ZIP_OUT = '/content/EmoInHindi_trained_model.zip'
with zipfile.ZipFile(ZIP_OUT, 'w', zipfile.ZIP_DEFLATED) as z:
    for name in os.listdir(OUTPUT_DIR):
        z.write(os.path.join(OUTPUT_DIR, name), arcname=name)
print('Saved model ZIP:', ZIP_OUT)
print('Files:', os.listdir(OUTPUT_DIR))

### Quick prediction test

In [ ]:
saved_bundle = joblib.load('/content/EmoInHindi_model/model_bundle.joblib')
def predict_emotions(text, bundle=saved_bundle):
    features = bundle['vectorizer'].transform([text])
    scores = bundle['model'].decision_function(features)[0]
    chosen = [label for label, score, threshold in zip(bundle['labels'], scores, bundle['thresholds']) if score >= threshold]
    return chosen

sample = 'मैं बहुत दुखी और परेशान हूँ।'
print('Input:', sample)
print('Predicted emotions:', predict_emotions(sample))

### Optional: download the trained ZIP

In [ ]:
from google.colab import files
files.download('/content/EmoInHindi_trained_model.zip')

## Notes for your report

- The CSV has 44,247 utterances across 1,814 dialogues and **17 distinct labels**. The README describes 16 labels; note this discrepancy in your report.
- Splits are grouped by dialogue ID to reduce leakage.
- Report the test metrics produced by this run; do not claim unmeasured scores.
- The model uses utterance text only. Context and emotion intensity are not modeled in this concise baseline.
- Use the dataset for research purposes only, as stated in its README.